# RegiBERT inference and benchmark notebook

[Open in Colab](https://colab.research.google.com/github/aogunleye/RegiBERT_French_Register_Classification/blob/main/inference.ipynb)

If you’re using Google Colab, enable a GPU accelerator (`Run` > `Change run type` > `GPU T4`)

0. Installation

In [1]:
%%capture
%pip install torch --extra-index-url https://download.pytorch.org/whl/cu121 
# or just %pip install torch if you don't have a GPU
%pip install transformers pandas scikit-learn matplotlib
# might take a very few minutes to install all dependencies so please be patient :)

1. Imports and config

In [39]:
import torch
from transformers import AutoTokenizer
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pandas as pd
import config
from src.model import RegiBERT
import sys
from pathlib import Path
project_root = Path.cwd()
src_path = project_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
from src.evaluate import evaluate_model, compute_metrics
from src.dataset import get_dataloaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

Device : cpu


2. Quick benchmark

By default, the cell below instantly displays the metrics recorded during the last training.
> **Important note:** If you want to rerun the full evaluation on the 45,654 examples in the validation set, set the parameter to `reevaluate=True`. 
> *On **Google Colab (with a GPU)**, this re-evaluation takes **a few seconds**. On a local CPU, it can take some minutes.*

In [ ]:
def show_benchmark(reevaluate=False):
    if reevaluate:
        print("Evaluation in progress...")
        
        _, val_loader, _ = get_dataloaders()
        criterion = torch.nn.KLDivLoss(reduction="batchmean")
        
        val_loss, preds, targets = evaluate_model(model, val_loader, criterion, device)
        mae_per_class, total_mae, accuracy, classes = compute_metrics(preds, targets)
        
        metrics_data = {"KL divergence": [f"{val_loss:.4f}"], "Global MAE": [f"{total_mae:.4f}"], "Accuracy": [f"{accuracy * 100:.2f}%"]}
        mae_data = {"Register": classes, "MAE": [f"{v:.4f}" for v in mae_per_class]}
    else:
        metrics_data = {"KL divergence": ["0.3654"], "Global MAE": ["0.1788"], "Accuracy": ["84.48%"]}
        mae_data = {"Registre": ["Soutenu", "Courant", "Familier"], "MAE": ["0.1088", "0.2596", "0.1679"]}

    print("Metrics overview:")
    display(pd.DataFrame(metrics_data))
    print("\nMAE per register:")
    display(pd.DataFrame(mae_data))

show_benchmark(reevaluate=False)

Metrics overview:


,KL Divergence,Global MAE,Accuracy
0,0.3654,0.1788,84.48%



MAE per register:


,Registre,MAE
0,Soutenu,0.1088
1,Courant,0.2596
2,Familier,0.1679


3. Loading

In [24]:
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
model = RegiBERT().to(device)

model.load_state_dict(torch.load("checkpoints/best_model.pt", map_location=device, weights_only=True))
model.eval()
print("RegiBERT model loaded successfully!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 530.19it/s]
[transformers] CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RegiBERT model loaded successfully!


4. Prediction function 

In [25]:
def predict_register(text):
    encoding = tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    
    with torch.no_grad():
        logprobs, _ = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.exp(logprobs).squeeze(0).cpu().numpy()
        
    labels = ["Soutenu", "Courant", "Familier"]
    
    print(f'"{text}"')
    for label, prob in zip(labels, probs):
        print(f" - {label}: {prob:.0%}\n")

    colors = ["#2b5c8f", "#4682b4", "#b0c4de"]
    plt.figure(figsize=(6, 2))
    plt.barh(["Soutenu", "Courant", "Familier"], probs, color=colors)
    plt.xlim(0, 1)
    plt.title(f'"{text}"', fontsize=10, italic=True)
    plt.tight_layout()
    plt.show()

5. Tests 

I invite you to test the function with other French sentences. 

Here are some examples:

In [ ]:
predict_register("J’peux moi aussi ouuuuuuu… que les p’tits ?")
predict_register("On a tous vu cette image en histoire-géo")
predict_register("La question de savoir si la vérité objective revient à la pensée humaine n'est pas une question théorique, mais une question pratique.")

"J’peux moi aussi ouuuuuuu… que les p’tits ?"
 - Soutenu: 0%

 - Courant: 3%

 - Familier: 97%

"On a tous vu cette image en histoire-géo"
 - Soutenu: 1%

 - Courant: 88%

 - Familier: 11%

"La question de savoir si la vérité objective revient à la pensée humaine n'est pas une question théorique, mais une question pratique."
 - Soutenu: 96%

 - Courant: 4%

 - Familier: 0%

